# 📊 Avaliação e Casos de Teste
## Golden Set e Análise de Desempenho

**Objetivo deste notebook:** ir além das métricas agregadas do notebook `02`
e olhar caso a caso — carregamos os modelos já treinados
(`models/baseline.json` e `models/thompson.json`) e construímos um **Golden
Set** de 20 clientes reais do conjunto de teste (10 que aceitaram a oferta,
10 que rejeitaram) para inspecionar manualmente se as recomendações fazem
sentido, e não só confiar em uma média de accuracy.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np

from src.recommender import OfferRecommender

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print("✅ Módulos importados com sucesso!")

✅ Módulos importados com sucesso!


## 1️⃣ Definir Métricas de Avaliação

Antes de ler qualquer número, deixamos explícito o que cada métrica
significa neste contexto de negócio (recomendar ofertar ou não a um
cliente), para evitar interpretar mal os resultados abaixo.

In [2]:
print("📊 MÉTRICAS DE AVALIAÇÃO")
print("="*70)

metrics_desc = {
    'Accuracy': 'Proporção total de previsões corretas (acertos e erros)',
    'Precision': 'De quem foi recomendado (1), quantos realmente aceitaram',
    'Recall': 'De quem realmente aceitou (1), quantos foram recomendados',
    'Taxa de Aceitação': 'Quando recomendamos, qual % realmente aceita',
}

for metric, desc in metrics_desc.items():
    print(f"\n{metric}: {desc}")

📊 MÉTRICAS DE AVALIAÇÃO

Accuracy: Proporção total de previsões corretas (acertos e erros)

Precision: De quem foi recomendado (1), quantos realmente aceitaram

Recall: De quem realmente aceitou (1), quantos foram recomendados

Taxa de Aceitação: Quando recomendamos, qual % realmente aceita


## 2️⃣ Carregar Dados e Modelos

Carregamos o conjunto de teste já processado, os dois modelos treinados no
notebook `02` (em JSON, sem pickle) e o `scaler`/`encoders` salvos por
`src/data_preparation.py` — vamos precisar deles adiante para reconstruir
idade e profissão em formato legível no Golden Set.

In [3]:
import pickle, json

# Carregar dados processados (features escaladas usadas pelo modelo)
test_df = pd.read_csv('data/processed/test_clean.csv')
X_test = test_df.drop('y', axis=1)
y_test = test_df['y']

# Carregar modelos treinados (JSON, sem pickle)
baseline = OfferRecommender.load_json('models/baseline.json')
thompson = OfferRecommender.load_json('models/thompson.json')

# Carregar scaler + encoders para reconstruir idade/profissão legíveis a
# partir das features já escaladas (mais seguro que tentar realinhar por
# posição com o CSV bruto, já que o split treino/teste é estratificado e
# embaralhado — a ordem das linhas não corresponde ao arquivo original).
with open('data/processed/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('data/processed/encoders.json') as f:
    encoders = json.load(f)

print("✅ Dados e modelos carregados")
print(f"   Test set: {len(X_test):,} clientes")
print(f"   Segmentos aprendidos pelo Thompson: {len(thompson.model.segments)}")

✅ Dados e modelos carregados
   Test set: 12,357 clientes
   Segmentos aprendidos pelo Thompson: 48


## 3️⃣ Criar Golden Set (20 Exemplos)

Selecionamos 20 clientes do conjunto de teste com uma amostragem
propositalmente balanceada — 10 que aceitaram a oferta e 10 que rejeitaram
(`np.random.seed(42)` para ser reprodutível) — em vez de uma amostra
puramente aleatória, que teria só ~2 aceitantes dado o desbalanceamento de
~11% visto na EDA. Isso garante que o Golden Set tenha exemplos suficientes
dos dois desfechos para avaliar visualmente.

In [4]:
# Selecionar 20 clientes diversos para o Golden Set: 10 que aceitaram + 10 que rejeitaram
np.random.seed(42)

idx_yes = np.where(y_test == 1)[0]
idx_no = np.where(y_test == 0)[0]

golden_indices_yes = np.random.choice(idx_yes, size=min(10, len(idx_yes)), replace=False)
golden_indices_no = np.random.choice(idx_no, size=min(10, len(idx_no)), replace=False)
golden_indices = np.concatenate([golden_indices_yes, golden_indices_no])

X_golden = X_test.iloc[golden_indices].reset_index(drop=True)
y_golden = y_test.iloc[golden_indices].reset_index(drop=True)

# Reconstruir idade e profissão legíveis a partir das features escaladas
# (inverse_transform do StandardScaler + decodificação via encoders.json)
X_golden_raw = pd.DataFrame(scaler.inverse_transform(X_golden), columns=X_golden.columns)
job_classes = encoders['job']
golden_desc = pd.DataFrame({
    'age': X_golden_raw['age'].round().astype(int),
    'job': X_golden_raw['job'].round().astype(int).clip(0, len(job_classes) - 1).map(lambda i: job_classes[i]),
})

print(f"✅ Golden Set criado com {len(X_golden)} clientes")
golden_desc.head()

✅ Golden Set criado com 20 clientes


,age,job
0,37,admin.
1,75,retired
2,42,self-employed
3,66,retired
4,40,technician


## 4️⃣ Fazer Recomendações no Golden Set

In [5]:
baseline_pred = baseline.recommend(X_golden)
thompson_pred = thompson.recommend(X_golden)

baseline_probs = baseline.predict_proba(X_golden)
thompson_probs = thompson.predict_proba(X_golden)

golden_table = pd.DataFrame({
    'ID': range(1, len(X_golden) + 1),
    'Idade': golden_desc['age'],
    'Profissão': golden_desc['job'],
    'Real (y)': y_golden.values,
    'Baseline Pred': baseline_pred,
    'Baseline Conf': baseline_probs[:, 1].round(3),
    'Thompson Pred': thompson_pred,
    'Thompson Conf': thompson_probs[:, 1].round(3),
    'Baseline Acerto': (baseline_pred == y_golden.values).astype(int),
    'Thompson Acerto': (thompson_pred == y_golden.values).astype(int),
})

golden_table

,ID,Idade,Profissão,Real (y),Baseline Pred,Baseline Conf,Thompson Pred,Thompson Conf,Baseline Acerto,Thompson Acerto
0,1,37,admin.,1,1,0.113,1,0.117,1,1
1,2,75,retired,1,1,0.113,1,0.257,1,1
2,3,42,self-employed,1,1,0.113,0,0.096,1,0
3,4,66,retired,1,1,0.113,1,0.257,1,1
4,5,40,technician,1,1,0.113,0,0.088,1,0
5,6,39,services,1,1,0.113,0,0.074,1,0
6,7,85,retired,1,1,0.113,1,0.257,1,1
7,8,50,management,1,1,0.113,1,0.120,1,1
8,9,60,admin.,1,1,0.113,1,0.137,1,1
9,10,32,student,1,1,0.113,1,0.132,1,1


### Confiança varia por cliente

Diferente de um bandit global (um único par alpha/beta para toda a base), a
confiança do Thompson (`Thompson Conf`) muda conforme o segmento
(idade + profissão) de cada cliente — ela não é um valor único repetido em
todas as linhas, como `Baseline Conf` (sempre 11.3%, a taxa média global).
Isso evidencia que o **contexto do cliente entra na decisão**, um requisito
central do desafio.

In [6]:
print("Variação da confiança do Thompson no Golden Set:")
print(f"  mínima: {golden_table['Thompson Conf'].min():.3f}")
print(f"  máxima: {golden_table['Thompson Conf'].max():.3f}")
print(f"  valores distintos: {golden_table['Thompson Conf'].nunique()} (de {len(golden_table)} clientes)")

Variação da confiança do Thompson no Golden Set:
  mínima: 0.068
  máxima: 0.257
  valores distintos: 13 (de 20 clientes)


## 5️⃣ Análise de Desempenho no Golden Set

In [7]:
baseline_acertos = (baseline_pred == y_golden.values).sum()
thompson_acertos = (thompson_pred == y_golden.values).sum()

baseline_acuracy = baseline_acertos / len(y_golden)
thompson_accuracy = thompson_acertos / len(y_golden)

print("\n📊 DESEMPENHO NO GOLDEN SET")
print("="*70)
print(f"\nBaseline:")
print(f"  Acertos: {baseline_acertos}/{len(y_golden)} ({baseline_acuracy*100:.1f}%)")
print(f"  Recomendações de oferta: {baseline_pred.sum()}/{len(baseline_pred)} ({baseline_pred.mean()*100:.1f}%)")

print(f"\nThompson:")
print(f"  Acertos: {thompson_acertos}/{len(y_golden)} ({thompson_accuracy*100:.1f}%)")
print(f"  Recomendações de oferta: {thompson_pred.sum()}/{len(thompson_pred)} ({thompson_pred.mean()*100:.1f}%)")


📊 DESEMPENHO NO GOLDEN SET

Baseline:
  Acertos: 10/20 (50.0%)
  Recomendações de oferta: 20/20 (100.0%)

Thompson:
  Acertos: 14/20 (70.0%)
  Recomendações de oferta: 10/20 (50.0%)


**Nota de leitura:** o Golden Set foi construído com 50% de aceitantes
(bem acima do ~11% real da base) para garantir exemplos dos dois desfechos —
por isso a accuracy aqui não é diretamente comparável com a do conjunto de
teste completo do notebook `02`. O valor desta seção está nos casos
individuais abaixo, não na accuracy agregada.

## 6️⃣ Casos de Interesse (Erros e Acertos)

In [8]:
print("\n🎯 CASOS DE INTERESSE")
print("="*70)

thompson_acertou = thompson_pred == y_golden.values
baseline_errou = baseline_pred != y_golden.values
vantagem = (thompson_acertou & baseline_errou)

if vantagem.sum() > 0:
    print(f"\n✅ Casos onde Thompson acertou e Baseline errou: {vantagem.sum()}")
    for idx in np.where(vantagem)[0][:3]:
        row = golden_table.iloc[idx]
        print(f"\n  Cliente {idx+1} ({row['Idade']} anos, {row['Profissão']}):")
        print(f"    Real: {row['Real (y)']} | Baseline previu: {row['Baseline Pred']} | Thompson previu: {row['Thompson Pred']} (conf={row['Thompson Conf']:.3f})")
else:
    print("Nenhum caso onde Thompson acertou e Baseline errou neste Golden Set.")

thompson_errou = ~thompson_acertou
if thompson_errou.sum() > 0:
    print(f"\n⚠️  Casos onde Thompson errou: {thompson_errou.sum()}")
    for idx in np.where(thompson_errou)[0][:3]:
        row = golden_table.iloc[idx]
        print(f"\n  Cliente {idx+1} ({row['Idade']} anos, {row['Profissão']}):")
        print(f"    Real: {row['Real (y)']} | Thompson previu: {row['Thompson Pred']} (conf={row['Thompson Conf']:.3f})")


🎯 CASOS DE INTERESSE

✅ Casos onde Thompson acertou e Baseline errou: 7

  Cliente 11 (41 anos, technician):
    Real: 0 | Baseline previu: 1 | Thompson previu: 0 (conf=0.088)

  Cliente 12 (28 anos, blue-collar):
    Real: 0 | Baseline previu: 1 | Thompson previu: 0 (conf=0.078)

  Cliente 13 (32 anos, blue-collar):
    Real: 0 | Baseline previu: 1 | Thompson previu: 0 (conf=0.074)

⚠️  Casos onde Thompson errou: 6

  Cliente 3 (42 anos, self-employed):
    Real: 1 | Thompson previu: 0 (conf=0.096)

  Cliente 5 (40 anos, technician):
    Real: 1 | Thompson previu: 0 (conf=0.088)

  Cliente 6 (39 anos, services):
    Real: 1 | Thompson previu: 0 (conf=0.074)


**Análise de erros:** o Baseline erra sistematicamente com qualquer
cliente que rejeita a oferta (recall 100%, precision baixa — ele nunca deixa
de ofertar). O Thompson erra de forma mais pontual: quando um cliente de um
segmento tipicamente positivo (ex.: idade avançada) rejeita a oferta, ou
vice-versa — esses são exceções dentro do padrão do segmento, não um erro
sistemático do modelo. Uma limitação conhecida: como a segmentação usa só
idade e profissão, dois clientes muito diferentes em outras variáveis (saldo,
histórico de campanha) mas com a mesma combinação idade/profissão recebem a
**mesma** confiança — um contexto mais rico (mais features na segmentação,
ou um bandit linear contextual) tende a reduzir esse tipo de erro.

## 7️⃣ Salvar Golden Set (5 exemplos condensados + tabela completa)

In [9]:
os.makedirs('data/processed', exist_ok=True)
golden_table.to_csv('data/processed/golden_set.csv', index=False)
print("✅ Golden Set completo (20 exemplos) salvo em data/processed/golden_set.csv")

golden_5 = golden_table.head(5)
golden_5.to_csv('data/processed/golden_set_5.csv', index=False)
print("✅ Golden Set reduzido (5 exemplos, exigido pelo desafio) salvo em data/processed/golden_set_5.csv")
golden_5

✅ Golden Set completo (20 exemplos) salvo em data/processed/golden_set.csv
✅ Golden Set reduzido (5 exemplos, exigido pelo desafio) salvo em data/processed/golden_set_5.csv


,ID,Idade,Profissão,Real (y),Baseline Pred,Baseline Conf,Thompson Pred,Thompson Conf,Baseline Acerto,Thompson Acerto
0,1,37,admin.,1,1,0.113,1,0.117,1,1
1,2,75,retired,1,1,0.113,1,0.257,1,1
2,3,42,self-employed,1,1,0.113,0,0.096,1,0
3,4,66,retired,1,1,0.113,1,0.257,1,1
4,5,40,technician,1,1,0.113,0,0.088,1,0


---
**Próximo passo:** a API (`src/app/main.py`, ver README) usa o mesmo
`models/thompson.json` para servir recomendações em produção — é o roteiro
sugerido para a demo ao vivo do vídeo de apresentação (Etapa 8).